# 08 可靠性、测试与演示整理

## 1. 本节点目标
让项目不仅在正常情况下能运行，也能在天气或模型服务失败时继续工作，并用自动化测试证明核心流程没有被后续修改破坏。

## 2. 完成结果与验收
- 无模型密钥时自动使用规则建议。
- 天气超时、返回异常和城市无效都有可理解的错误或降级结果。
- SQLite 测试使用临时数据库，不会修改正式数据。
- 70 项自动化测试全部通过。
- README、架构说明和三分钟演示流程已经补齐。

## 3. 本节点文件结构
- `tests/`：数据库、规则、天气、工具、Agent、账户和外观测试。
- `src/smart_laundry/weather.py`：网络超时、响应转换和天气缓存。
- `src/smart_laundry/recommendations.py`：不依赖模型的规则计划。
- `src/smart_laundry/agent.py`：模型异常后的降级和最大轮数。
- `docs/demo.md`：可重复的演示步骤。
- `docs/architecture.md`：模块边界和数据流。

## 4. 关键代码解释
测试不会访问真实天气或真实模型，而是把固定返回值注入业务代码。下面的小例子展示了测试中最重要的思想：同一个输入应得到确定结果。

In [ ]:
def is_due(elapsed_days, interval_days):
    return elapsed_days >= interval_days

assert is_due(30, 30) is True
assert is_due(29, 30) is False

## 5. 数据流
用户请求 → 尝试取得天气和物品事实 → 调用模型生成计划 → 若缺少密钥或调用失败则进入规则引擎 → 页面标注当前模式并展示结果。数据库测试则使用临时路径：创建临时库 → 执行操作 → 断言结果 → 测试结束自动清理。

## 6. 关键概念
- **降级**：外部能力不可用时，保留核心功能。
- **Mock/Fake**：用可控制的替代对象模拟外部服务。
- **测试隔离**：每个测试拥有独立数据，不依赖运行顺序。
- **回归测试**：修改代码后再次执行，检查旧功能是否被破坏。

## 7. 为什么这样设计
天气 API 和大模型都可能受网络、额度或供应商影响。如果核心建议完全依赖它们，演示会很脆弱。因此规则建议是独立能力，AI 是增强层，而不是应用能否启动的前提。

## 8. 常见错误与排查
- 页面无法启动：先确认已激活 `smart-laundry` 环境并安装依赖。
- 天气一直失败：检查城市名称和网络，再观察是否已进入规则降级。
- 测试修改正式数据：确认测试使用 `tmp_path`，没有读取开发数据库路径。
- Streamlit 重复写入：检查按钮状态和同日幂等逻辑。

## 9. 面试可能追问
**问：为什么测试不直接调用真实 API？**
答：真实网络会让测试变慢且不稳定，也可能消耗额度。业务正确性应由固定输入验证，真实 API 只用于少量手动验收。

**追问：降级模式如何避免误导？**
答：页面明确标注规则模式，天气缺失时不编造天气，只按物品超期情况给建议。

## 10. 必须掌握的最少知识
理解断言、临时数据库、模拟外部服务和降级四个概念；能够运行 `python -m pytest`，并从失败测试名称定位到对应模块。

## 11. 可自测小题
1. 为什么天气测试不应依赖当天真实天气？
2. 无模型密钥时应用应怎样表现？
3. 临时数据库解决了什么问题？
4. 达到 Agent 最大轮数后为什么要安全停止？

<details><summary>参考答案</summary>真实天气不可重复；进入明确的规则模式；避免测试污染正式数据；防止无限工具循环。</details>

## 12. 动手小练习
1. 为一个新的周期边界补充测试，例如 59% 与 60%。
2. 把 Fake 天气改成雨天，观察规则建议是否暂缓户外晾晒。

## 13. 本节点术语表
- `pytest`：Python 测试工具。
- `fixture`：测试所需的可重复准备数据。
- `timeout`：外部请求允许等待的最长时间。
- `fallback`：主要方案失败后的备用方案。

## 14. 下一节点连接
可靠性和测试结果会成为简历中的可验证证据。下一节点将整理项目描述、架构讲解、演示话术、诚实边界和 GitHub 发布材料。